# R19-H227 - Probe-pair hygiene: the cross-pairing audit

**Hypothesis** - auditing every gold in the canonical probe sets (`probes-wide-h188`, `probes-wide-v2-h195`, `probes-conflict-h204`) for source-doc consistency finds **< 5% cross-paired golds** - the H191 case being a residue-set artifact, not systemic.

**Method** - deterministic string audit (H190-normalized) of each gold against its CITED document's pymupdf text layer; if absent there, search every other corpus document and record where it appears. No LLM. CPU-only. neo4j2 READ-ONLY (fingerprint-asserted) only if a repair forces a census re-run.

**Bar** - < 5% cross-paired confirms; >= 5% refutes and flags re-adjudication.

## Imports

In [1]:
import os, re, json, itertools, unicodedata, datetime, hashlib
from pathlib import Path
from collections import defaultdict
os.environ["CUDA_VISIBLE_DEVICES"] = ""            # CPU-only
import fitz                                         # pymupdf text layer
from dotenv import dotenv_values
from neo4j import GraphDatabase                     # only for the (conditional) fingerprint-asserted census
from rich import print as rprint

## Configuration - paths, H190 normalizers, unified presence test

In [2]:
ROOT = Path("..")
PDFDIR = ROOT / "data/external/cpap-datasheets-and-manuals"
PROBE_SETS = ["probes-wide-h188", "probes-wide-v2-h195", "probes-conflict-h204"]
CANONICAL = {"probes-wide-h188", "probes-wide-v2-h195"}      # conflict set is derived
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NEO4J2 = "bolt://172.19.0.9:7687"                            # pinned baseline, READ-ONLY
EXPECT_RENDER_FP = "96ab16d299fbbc71"
LOG = ROOT / "logs/h226-h227-forensics.log"
def log(msg):
    line = f"{datetime.datetime.now(datetime.timezone.utc).isoformat()} [H227] {msg}"
    with open(LOG, "a") as f: f.write(line + "\n")

# ---- H190 normalizers (verbatim family from code_arm_h206 / pixel_forensics_h217) ----
DASH = {"\u2013":"-","\u2014":"-","\u2011":"-","\u2212":"-","~":"-"}
_TM = dict.fromkeys(map(ord, "\u00ae\u2122\u00a9"), None)
def basenorm(s):
    s = (s or "").translate(_TM)
    for k, v in DASH.items(): s = s.replace(k, v)
    s = unicodedata.normalize("NFKC", s).replace("\u00d7","x").replace("\u00b7","x")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)          # 1,130 -> 1130
    s = re.sub(r"\bto\b", "-", s)                  # "4 to 20" -> "4-20"
    return re.sub(r"\s+", " ", s.casefold()).strip()
def canon(s):
    s = basenorm(s); s = re.sub(r"h\s*2?\s*o", "h2o", s)   # h 2 o / ho / h2o -> h2o
    return re.sub(r"\s+", "", s)
UNITS = r"(cmh2o|cmho|hpa|kpa|va|lpm|mm|cm|dba|db|kg|g|oz|ml|l|w|hz|min|m)"
def unitstrip(s): return re.sub(UNITS, "", canon(s))
def codenorm(s):
    s = s or ""
    for k, v in {"\u2122":"","\u00ae":"","\u00a9":"","\ufb01":"fi","\ufb02":"fl"}.items(): s = s.replace(k, v)
    s = unicodedata.normalize("NFKD", s); s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s\-]", "", s).casefold()
def is_code(g):
    if re.search(r"h ?2? ?o|hpa|cm\b|-\s*\d", basenorm(g)): return False
    cn = codenorm(g); return bool(cn) and re.match(r"^[a-z]{0,2}\d{3,}[a-z0-9]*$", cn) is not None
def dim_nums(gold):
    return re.findall(r"\d+(?:\.\d+)?", basenorm(gold)) if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", basenorm(gold)) else None
rprint(f"[cyan]config[/cyan] sets={PROBE_SETS} corpus={PDFDIR.name} stamp={STAMP} (CPU-only)")

config sets=['probes-wide-h188', 'probes-wide-v2-h195', 'probes-conflict-h204'] corpus=cpap-datasheets-and-manuals 
stamp=20260707T204721Z (CPU-only)

## Data loading - pymupdf text layers (all corpus documents) + probe sets

In [3]:
DOC = {}
for p in sorted(PDFDIR.glob("*.pdf")):
    t = "\n".join(pg.get_text() for pg in fitz.open(p))
    DOC[p.name] = dict(g=basenorm(t), cx=canon(t), ux=unitstrip(t), code=codenorm(t))
def load(fn): return json.load(open(ROOT / "data/processed" / f"{fn}.json"))["probes"]
SETS = {fn: load(fn) for fn in PROBE_SETS}
rprint(f"[green]indexed[/green] {len(DOC)} document text layers | " +
       " ".join(f"{k}={len(v)}p" for k, v in SETS.items()))

MuPDF error: format error: No default Layer config



indexed 28 document text layers | probes-wide-h188=101p probes-wide-v2-h195=219p probes-conflict-h204=15p

## Deterministic presence test (H190-normalized)

A gold is *present* in a document text layer if, after H190 normalization, it matches by:
dimension permutation (`AxBxC`, unit-agnostic), code token (`codenorm` substring), pressure/flow range (`a-b`, separator- and H2O-folded), value+unit adjacency, or prose substring / >=0.6 token overlap.

In [4]:
def present(gold, name):
    d = DOC[name]; g = basenorm(gold)
    dn = dim_nums(gold)
    if dn and len(dn) == 3:
        return any((a+"x"+b+"x"+c) in d["ux"] for a, b, c in itertools.permutations(dn))
    if is_code(gold):
        cg = codenorm(gold); return len(cg) >= 4 and cg in d["code"]
    m = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)", g)
    if m and re.search(r"\d\s*-\s*\d.*(cm|hpa|kpa|h ?2? ?o|ho|va|lpm|l/?min)", g):
        return (m.group(1)+"-"+m.group(2)) in d["cx"]
    vu = re.findall(r"(\d[\d\.]*)\s*"+UNITS+r"\b", canon(gold).replace("h2o", " h2o"))
    if vu: return all((num+unit) in d["cx"] for num, unit in vu)
    cg = canon(gold)
    if re.fullmatch(r"[\d\.]+h2o?", cg) or re.fullmatch(r"\d[\d\.]*", cg):
        return cg in d["cx"]
    if g and g in d["g"]: return True
    w = set(re.findall(r"[a-z][a-z0-9\-]{2,}", g)); cw = set(re.findall(r"[a-z][a-z0-9\-]{2,}", d["g"]))
    return bool(w) and len(w & cw) / len(w) >= 0.6
def audit_gold(gold, cited):
    cp = cited in DOC and present(gold, cited)
    return cp, ([] if cp else [dn for dn in DOC if dn != cited and present(gold, dn)])

## Cross-pairing audit - every gold vs its cited document, then vs all others

In [5]:
rows_by = {}
for sn, probes in SETS.items():
    rows = []
    for p in probes:
        pid = p.get("id")
        if sn == "probes-conflict-h204":                       # each value cited to its paired doc
            for i, v in enumerate(p["values"]):
                cited = p["doc_pair"][i] if i < len(p["doc_pair"]) else p["doc_pair"][0]
                cp, el = audit_gold(v, cited)
                rows.append(dict(id=f"{pid}.{i}", gold=v, cited_doc=cited, found_in_cited=cp, found_elsewhere=el))
        else:
            for g in p["gold_evidence"]:
                cp, el = audit_gold(g, p["source_document"])
                rows.append(dict(id=pid, gold=g, cited_doc=p["source_document"], found_in_cited=cp, found_elsewhere=el))
    rows_by[sn] = rows

audit = {}
for sn, rows in rows_by.items():
    n = len(rows)
    cross = [r for r in rows if not r["found_in_cited"] and r["found_elsewhere"]]
    absent = [r for r in rows if not r["found_in_cited"] and not r["found_elsewhere"]]
    audit[sn] = dict(n=n, in_cited=n-len(cross)-len(absent), cross_paired=len(cross),
                     absent_everywhere=len(absent), cross_rate=len(cross)/n, cross_rows=cross)
    rprint(f"[bold]{sn}[/bold]: {n} golds | in-cited {n-len(cross)-len(absent)} | "
           f"[yellow]CROSS-PAIRED {len(cross)} ({len(cross)/n:.1%})[/] | absent-everywhere {len(absent)}")
    for r in cross:
        rprint(f"   CROSS {r['id']:7s} \"{r['gold'][:28]}\" cited={r['cited_doc'][:34]} -> {[d[:26] for d in r['found_elsewhere']][:3]}")
    log(f"{sn}: n={n} cross={len(cross)} ({len(cross)/n:.1%}) absent={len(absent)}")

probes-wide-h188: 101 golds | in-cited 101 | CROSS-PAIRED 0 (0.0%) | absent-everywhere 0

probes-wide-v2-h195: 219 golds | in-cited 218 | CROSS-PAIRED 1 (0.5%) | absent-everywhere 0

CROSS V191    "Bluetooth connectivity" cited=Sleep And Respiratory Medical Devi -> 
['BC-Dreamstation-Standard-C', 'DSDC-CPAP-Therapy-Catalogu', 'DreamStation_CPAP_Pro_Data']

probes-conflict-h204: 30 golds | in-cited 30 | CROSS-PAIRED 0 (0.0%) | absent-everywhere 0

## Audit table (full) and flagged-case adjudication

The single flagged case in the canonical sets is adjudicated by reading the cited document directly.

In [6]:
full_table = [dict(set=sn, **{k: r[k] for k in ("id","gold","cited_doc","found_in_cited","found_elsewhere")})
              for sn, rows in rows_by.items() for r in rows]
rprint(f"[green]full audit table[/green] {len(full_table)} gold rows across {len(SETS)} sets")

# adjudicate V191 ("Bluetooth connectivity" / Philips DreamStation)
adjud = {}
for r in audit["probes-wide-v2-h195"]["cross_rows"]:
    if r["id"] == "V191":
        cited_txt = re.sub(r"\s+", " ", "\n".join(pg.get_text() for pg in fitz.open(PDFDIR / r["cited_doc"]))).lower()
        bt = cited_txt.count("bluetooth")
        adjud["V191"] = dict(cited_bluetooth_mentions=bt,
            note="cited doc states 'integrated bluetooth connects with dreammapper' / 'built-in bluetooth' in the DreamStation ordering block (product named 7x); the FEATURE is present for the product in the cited doc - the flag is a verbatim-paraphrase gap ('connectivity' not literal), NOT a cross-pairing/ownership reassignment",
            genuine_cross_pairing=False)
        rprint(f"[cyan]V191 adjudication[/cyan] cited-doc 'bluetooth' mentions={bt} -> genuine cross-pairing: [bold]False[/bold] (verbatim-paraphrase artifact)")

full audit table 350 gold rows across 3 sets

V191 adjudication cited-doc 'bluetooth' mentions=3 -> genuine cross-pairing: False (verbatim-paraphrase artifact)

## Bar evaluation + repair decision

In [7]:
genuine_cross = {}
for sn, a in audit.items():
    g = a["cross_paired"]
    if sn == "probes-wide-v2-h195":
        g -= sum(1 for r in a["cross_rows"] if not adjud.get(r["id"], {}).get("genuine_cross_pairing", True))
    genuine_cross[sn] = g
canon_golds = sum(audit[s]["n"] for s in CANONICAL)
canon_cross_flagged = sum(audit[s]["cross_paired"] for s in CANONICAL)
canon_cross_genuine = sum(genuine_cross[s] for s in CANONICAL)
rate_flagged = canon_cross_flagged / canon_golds
rate_genuine = canon_cross_genuine / canon_golds
BAR = 0.05
verdict = "CONFIRMED" if rate_flagged < BAR else "REFUTED"
rprint(f"[bold]canonical cross-paired[/bold] flagged {canon_cross_flagged}/{canon_golds}={rate_flagged:.2%} | "
       f"genuine {canon_cross_genuine}/{canon_golds}={rate_genuine:.2%} | bar<{BAR:.0%} -> [bold]{verdict}[/bold]")

# repair pass: only genuine cross-pairings need corrected/removed golds
repairs = []                     # zero genuine cross-pairings -> no repaired files written
census_rerun = bool(repairs)
rprint(f"[green]repairs[/green] {len(repairs)} (no frozen set overwritten) | census re-run required: {census_rerun}")
log(f"BAR flagged={rate_flagged:.4f} genuine={rate_genuine:.4f} verdict={verdict} repairs={len(repairs)} census_rerun={census_rerun}")

canonical cross-paired flagged 1/320=0.31% | genuine 0/320=0.00% | bar<5% -> CONFIRMED

repairs 0 (no frozen set overwritten) | census re-run required: False

## Pinned census re-run (conditional)

The H207 census reads `probes-wide-h188`. It re-runs only if a wide/h188 gold was repaired. Zero canonical golds were cross-paired, so no gold was repaired and the census re-run is **skipped**. For provenance we still assert (READ-ONLY) that neo4j2 is the pinned H207 baseline via the render + graph fingerprints.

In [8]:
fp_assert = {}
env = dotenv_values(ROOT / ".env"); AUTH = ("neo4j", env["NEO4J_PASSWORD"])
dr = GraphDatabase.driver(NEO4J2, auth=AUTH)
with dr.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id,e.name AS name,e.description AS description,properties(e) AS props,labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id RETURN DISTINCT a.id AS a,b.id AS b,type(r) AS rel").data()
    alias_rows = s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id RETURN e.id AS eid, collect(DISTINCT a.id)[..5] AS aliases").data()
    emb_head = {r["id"]: r["head"] for r in s.run("MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()}
dr.close()
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
alias_by = {r["eid"]: r["aliases"] for r in alias_rows}
rels_by = defaultdict(list)
for e in edges: rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))
def spec_of(r): return {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r = node[nid]; spec = dict(spec_of(r))
    for a in [a for a in alias_by.get(nid, []) if a in node]:
        for k, v in spec_of(node[a]).items(): spec.setdefault(k, v)
    return spec
def base_render(nid):
    r = node[nid]; spec = merged_spec(nid); al = [a for a in alias_by.get(nid, []) if a in node]
    aka = (f"Also known as: {', '.join(names.get(a,'') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid):
    rels = "; ".join(f"{t} -> {names.get(b,'')}" for t, b in rels_by.get(nid, [])[:15])
    return base_render(nid) + "\nRelations: " + rels
CANON_SPEC = dict(source="pipeline._retrieve_local entity_blocks (production query path)",
    per_seed=["## name (types)","Also known as (SAME_AS*1..2, <=5)","description","Properties: json(prop_* keys, alias-merged)","Relations: type -> name (<=15, non-SIMILAR_TO)"],
    proposition_channel="per-seed attached propositions (Proposition-[:ABOUT]->seed)",
    eval_k=64, retrieve_top_k=128, rel_limit=15,
    seed_order="score desc, id asc (deterministic tie-break)",
    scorer="deterministic: exact_present(numeric/code) OR word_overlap>=0.6(prose); CPU-only, no NLI")
ALL = sorted(node)
render_fp = hashlib.sha256((json.dumps({k: CANON_SPEC[k] for k in sorted(CANON_SPEC)}, default=str) + "\x1e" +
    "\x1e".join(seed_render(c) for c in ALL)).encode()).hexdigest()[:16]
rb = "\x1e".join(seed_render(c) for c in ALL)
eb = ";".join(f"{c}:" + ",".join(f"{x:.4f}" for x in emb_head.get(c, [])) for c in ALL)
graph_fp = dict(node_count=len(node), edge_count=len(edges), embedding_count=len(emb_head),
    content_hash=hashlib.sha256(rb.encode()).hexdigest()[:16], embedding_digest=hashlib.sha256(eb.encode()).hexdigest()[:16])
assert render_fp == EXPECT_RENDER_FP, f"render fingerprint drift: {render_fp} != {EXPECT_RENDER_FP}"
fp_assert = dict(render_fingerprint=render_fp, render_match=True, graph_fingerprint=graph_fp)
rprint(f"[magenta]fingerprint asserted[/magenta] render={render_fp} (==expected) graph={graph_fp}")
rprint("[bold]census re-run SKIPPED[/bold] - zero canonical golds cross-paired, no gold repaired")
log(f"fingerprint render={render_fp} match=True graph={graph_fp}; census SKIPPED")

fingerprint asserted render=96ab16d299fbbc71 (==expected) graph={'node_count': 2798, 'edge_count': 3905, 
'embedding_count': 2798, 'content_hash': '6fdc41bde495d1a3', 'embedding_digest': '2a3908456d2e2d8c'}

census re-run SKIPPED - zero canonical golds cross-paired, no gold repaired

## Machine-readable report

In [9]:
report = dict(
    hypothesis="R19-H227", stamp=STAMP, driver=NEO4J2, mode="READ-ONLY (census skipped)",
    bar=dict(threshold=BAR, metric="canonical cross-paired share"),
    per_set={sn: {k: audit[sn][k] for k in ("n","in_cited","cross_paired","absent_everywhere","cross_rate")} for sn in SETS},
    genuine_cross_paired=genuine_cross,
    canonical=dict(golds=canon_golds, cross_flagged=canon_cross_flagged, cross_genuine=canon_cross_genuine,
                   rate_flagged=rate_flagged, rate_genuine=rate_genuine),
    verdict=verdict, flagged_adjudication=adjud, repairs=repairs, repaired_files=[],
    census_rerun=census_rerun, census_note="skipped - zero canonical golds cross-paired",
    fingerprint=fp_assert, full_audit_table=full_table)
out = ROOT / "reports" / f"probe-hygiene-h227-{STAMP}.json"
out.write_text(json.dumps(report, indent=1))
rprint(f"[green]report written[/green] {out}")
log(f"report {out.name} verdict={verdict}")

report written ../reports/probe-hygiene-h227-20260707T204721Z.json